In [ ]:
using MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8433*2*pi * (rand(1)[1]/5+0.9)
ω_b = 2*ω_a

φ_a = 0.1123
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 36.924*2*pi*1.5#*(rand(1)[1]/4+0.875)


##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

ϵ_0 = 0.5013#0.6013
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.124276*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper


system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, 0.0)
environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)
environment_prepared = prepare(system,environment)
dt = get_dt(system, environment)

In [ ]:
ϵ_0_list = vcat(0.5:.05:1,1:.15:0.6pi);
im_a_0_list = -6:0.5:6;
t_max = 60/κ_b
t_min = 0.0
t_span = (t_min, t_max)
Shannon_entropy_matrix = Array{Float64}(undef, length(im_a_0_list), length(ϵ_0_list))
for (i,ϵ_0) in ProgressBar(enumerate(ϵ_0_list))
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10.124276*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    for (j,im_a_0) in enumerate(im_a_0_list)

        re_a_0, im_a_0 = 0, im_a_0
        ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
        ϕ_b_0, q_b_0=0.0, 0.0

        system = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
        system_prepared = prepare(system)
        u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        t_save = 30/κ_b:dt:60/κ_b
        ϕ_a, _, q_a, _, t_list = default_solver((u_dot_0, u_0), t_span, p, dt; equation=default_equation!, t_save = t_save)

        ω =  ω_a
        Re = ϕ_a / sqrt(2*Z_a)
        Im = q_a * sqrt(Z_a/2);
        Re_a =@. Re * cos(ω * t_list) + Im * sin(-ω * t_list)
        Im_a =@. -Re * sin(-ω * t_list) + Im * cos(ω * t_list)
        F, freqs = FFT_perso(Re_a.-sum(Re_a)/length(Re_a), t_list)
        power_density = abs.(F).^2
        power_density_normalized = power_density ./sum(power_density)
        Shannon_entropy = -sum(power_density_normalized.*log.(power_density_normalized))
        Shannon_entropy_matrix[j,i] = Shannon_entropy
    end
end
        

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b

heatmap(g2_list/2/pi*1e3,im_a_0_list,Shannon_entropy_matrix)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\Im[a(0)]",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(im_a_0_list),maximum(im_a_0_list)),
    colorbar_title="\n"*"Shannon entropy", title = "Mean field ",
    clims = (0, maximum(Shannon_entropy_matrix)),
    right_margin=10mm)
#savefig("../../../figures/chaos/Shannon_entropy_02_12_2025.pdf")

In [ ]:
ϵ_0_list = vcat(0.6:.05:1)#,1:.15:0.6pi);
im_a_0_list = -6:0.5:6;
t_max = 40/κ_b
t_min = 0.0
t_span = (t_min, t_max)
Shannon_entropy_matrix = Array{Float64}(undef, length(im_a_0_list), length(ϵ_0_list))
n_iter = 10

ω_a_list = 3.8433*2*pi*(rand(n_iter) ./ 5 .+ 0.9)

In [ ]:

Threads.@threads for i in 1:length(ϵ_0_list)
    ϵ_0 = ϵ_0_list[i]
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10.124276*g2
    for index in 1:n_iter
        ω_a = ω_a_list[index]
        ω_b = 2*ω_a

        φ_a = 0.1123
        Z_a = 2*ϕ_quantum^2*φ_a^2
        C_a_inv = Z_a*ω_a
        E_Ca = C_a_inv/2

        φ_b = 0.204
        Z_b = 2*ϕ_quantum^2*φ_b^2
        C_b_inv = Z_b*ω_b
        E_Cb = C_b_inv/2

        ##Drive
        ω_d=ω_b
        drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
        drive_prepared = prepare(drive)
        for (j,im_a_0) in enumerate(im_a_0_list)

            re_a_0, im_a_0 = 0, im_a_0
            a_0_j = re_a_0 + 1im*im_a_0
            b_0=0.0

            system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)
            u_0_tot = get_init(system_prepared, environment_prepared)

            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            t_save = 10/κ_b:dt:t_max
            a, _, t_list = default_solver(u_0_tot, t_span, p, dt; equation=default_equation!, t_save = t_save)

            F, freqs = FFT_perso(real.(a), t_list)
            power_density = abs.(F).^2
            power_density_normalized = power_density ./sum(power_density)
            Shannon_entropy = -sum(power_density_normalized.*log.(power_density_normalized))
            Shannon_entropy_matrix[j,i] = Shannon_entropy/n_iter
        end
    end
end     

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b

heatmap(g2_list/2/pi*1e3,im_a_0_list,Shannon_entropy_matrix)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\Im[a(0)]",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(im_a_0_list),maximum(im_a_0_list)),
    colorbar_title="\n"*"Shannon entropy", title = "Mean field ",
    clims = (0, maximum(Shannon_entropy_matrix)),
    right_margin=10mm)
#savefig("../../../figures/chaos/Shannon_entropy_02_12_2025.pdf")